# Module 0 - See Your Goal: a Tiny Model Writes

## What you'll build

Before you build anything, let's see where this is all heading.

We've already trained a tiny GPT for you on a small slice of
Shakespeare and saved its weights to a compact file. In this module you
load those weights and watch the model **write text, one character at a
time**, right here in your browser -- no GPU, no PyTorch, just NumPy.

How is that possible? A trained model is just a big pile of numbers (the
weights) plus a fixed recipe for combining them (the forward pass). The
recipe is the same arithmetic whether you run it in PyTorch on a GPU or
in plain NumPy on a laptop. We saved the weights as a `.npz` file and
re-implemented the exact forward pass in NumPy in
`reference/demo/numpy_forward.py`. You can read it side by side with the
PyTorch model later and see they compute the same thing.

**Generation** works like autocomplete: give the model a starting
character, it predicts the most likely next character, you append it,
and repeat. Because we always take the single most likely character
("greedy" decoding), the output is fully deterministic -- the same
start always produces the same continuation.

Everything in this course builds up to understanding every line of that
forward pass. For now, just run it and enjoy the magic.

In [ ]:
# TODO: load the demo weights + metadata and generate text.
# Steps (fill in where marked):
#   1. load book/_static/demo_meta.json (gives stoi / itos / config)
#   2. load book/_static/demo_weights.npz (the trained fp16 weights)
#   3. build a GPTConfig from the saved config dict
#   4. call reference.demo.numpy_forward.generate from a start character
import json, os
import numpy as np

STATIC = os.path.join('..', 'book', '_static')

# TODO: set `text` to the generated string starting from the letter 'F'.
text = None
raise NotImplementedError


## Reveal the reference solution

Run the hidden cell to load the weights and generate. It defines `text`
(the generated string) which the check below inspects.

In [ ]:
# Reference solution: load demo weights + meta, run the NumPy forward pass.
import json, os
import numpy as np
from reference.gpt.model import GPTConfig
from reference.demo.numpy_forward import generate

STATIC = os.path.join('..', 'book', '_static')
with open(os.path.join(STATIC, 'demo_meta.json'), encoding='utf-8') as f:
    meta = json.load(f)
stoi = meta['stoi']
itos = {int(k): v for k, v in meta['itos'].items()}
c = meta['config']
cfg = GPTConfig(n_layer=c['n_layer'], n_head=c['n_head'], n_embd=c['n_embd'],
                block_size=c['block_size'], vocab_size=c['vocab_size'],
                dropout=c['dropout'])

loaded = np.load(os.path.join(STATIC, 'demo_weights.npz'))
weights = {k: loaded[k].astype(np.float32) for k in loaded.files}

start = np.array([[stoi['F']]])
out = generate(weights, start, cfg, max_new_tokens=200)
text = ''.join(itos[i] for i in out[0].tolist())
print(text)


## Check your work

The check confirms the model actually produced text and that generation
is deterministic: running the same greedy generation from the same start
character twice gives the exact same string.

In [ ]:
# Self-contained check: output is non-empty and deterministic.
import json, os
import numpy as np
from reference.gpt.model import GPTConfig
from reference.demo.numpy_forward import generate

with open(os.path.join(STATIC, 'demo_meta.json'), encoding='utf-8') as f:
    _meta = json.load(f)
_stoi = _meta['stoi']
_itos = {int(k): v for k, v in _meta['itos'].items()}
_c = _meta['config']
_cfg = GPTConfig(n_layer=_c['n_layer'], n_head=_c['n_head'], n_embd=_c['n_embd'],
                 block_size=_c['block_size'], vocab_size=_c['vocab_size'],
                 dropout=_c['dropout'])
_loaded = np.load(os.path.join(STATIC, 'demo_weights.npz'))
_w = {k: _loaded[k].astype(np.float32) for k in _loaded.files}

def _gen():
    s = np.array([[_stoi['F']]])
    o = generate(_w, s, _cfg, max_new_tokens=40)
    return ''.join(_itos[i] for i in o[0].tolist())

_a, _b = _gen(), _gen()
assert text is not None and len(text) > 0, 'no text generated'
assert _a == _b, 'generation is not deterministic'
assert all(ch in _itos.values() for ch in _a), 'generated chars out of vocab'
print('PASS', f'score=1.00  (generated {len(text)} chars, deterministic)')
